# ChatPPT v0.2 
本版本目标：
- 使用 python-pptx 自动生成 PowerPoint 文件，内容包括文本、图片、表格和图表 。
- 使用 Gradio 搭建一个 ChatBot 作为图形化用户界面（GUI），支持将用户输入转换为 ChatPPT PowerPoint 标准输入格式（Markdown），并最终生成 PowerPoint 文件。

In [7]:
import os
import re
import sys
from pathlib import Path

import gradio as gr
from pptx import Presentation
from pptx.util import Inches
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE

SRC_DIR = Path('src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from content_formatter import ContentFormatter
from input_parser import parse_input_text
from ppt_generator import generate_presentation
from template_manager import load_template, get_layout_mapping
from layout_manager import LayoutManager

PROMPT_FILE = Path('prompts/content_formatter.txt')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_PATH = Path('templates/Fair frames presentation.pptx')
if not TEMPLATE_PATH.exists():
    TEMPLATE_PATH = Path('templates/MasterTemplate.pptx')
if not TEMPLATE_PATH.exists():
    raise FileNotFoundError('No template found under templates/.')

ppt_template = load_template(str(TEMPLATE_PATH))
layout_manager = LayoutManager(get_layout_mapping(ppt_template))

try:
    formatter = ContentFormatter(prompt_file=str(PROMPT_FILE))
    FORMATTER_READY = True
except Exception as e:
    formatter = None
    FORMATTER_READY = False
    print('Formatter init failed, fallback mode will be used:', e)

print('Local src:', SRC_DIR)
print('Template path:', TEMPLATE_PATH)
print('Formatter ready:', FORMATTER_READY)

Formatter init failed, fallback mode will be used: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
Local src: /Users/elon/code/agents-bootcamp/chatppt/src
Template path: templates/Fair frames presentation.pptx
Formatter ready: False


In [8]:
def fallback_markdown(user_input: str) -> str:
    """当 LLM 不可用时的最小兜底格式。"""
    lines = [x.strip('- ').strip() for x in user_input.splitlines() if x.strip()]
    bullets = [f'- {x}' for x in lines[:4]] or ['- 内容要点 1', '- 内容要点 2']
    return '\n'.join([
        '# 自动生成演示',
        '',
        '## 概览',
        *bullets,
        '',
        '## 图片页',
        '![示例图片](images/performance_chart.png)',
    ])


def format_to_chatppt_markdown(user_input: str) -> str:
    """优先使用 chatppt-main ContentFormatter + prompts/content_formatter.txt。"""
    if formatter is not None:
        try:
            md = formatter.format(user_input)
            if isinstance(md, str) and '# ' in md and '## ' in md:
                return md
        except Exception as e:
            print('Formatter runtime failed, use fallback:', e)
    return fallback_markdown(user_input)


def extract_table_and_chart_blocks(md_text: str):
    """最小补充：提取 markdown 中的表格块和 chart 代码块。"""
    sections = []
    lines = md_text.splitlines()
    i = 0
    current_title = None

    while i < len(lines):
        line = lines[i].strip()

        if line.startswith('## '):
            current_title = line[3:].strip()
            i += 1
            continue

        if line.startswith('|'):
            table_lines = []
            while i < len(lines) and lines[i].strip().startswith('|'):
                table_lines.append(lines[i].strip())
                i += 1
            if len(table_lines) >= 2:
                sections.append({'type': 'table', 'title': current_title or '表格', 'lines': table_lines})
            continue

        if line.startswith('```chart'):
            i += 1
            chart_lines = []
            while i < len(lines) and not lines[i].strip().startswith('```'):
                chart_lines.append(lines[i].strip())
                i += 1
            sections.append({'type': 'chart', 'title': current_title or '图表', 'lines': chart_lines})
            i += 1
            continue

        i += 1

    return sections


print('Formatter + extractor ready.')

Formatter + extractor ready.


In [ ]:
def _append_table_slide(prs: Presentation, title: str, table_lines):
    headers = [x.strip() for x in table_lines[0].strip('|').split('|')]
    rows = []
    for row_line in table_lines[2:]:
        rows.append([x.strip() for x in row_line.strip('|').split('|')])

    slide = prs.slides.add_slide(prs.slide_layouts[1] if len(prs.slide_layouts) > 1 else prs.slide_layouts[0])
    if slide.shapes.title:
        slide.shapes.title.text = title

    if not headers:
        return

    shape = slide.shapes.add_table(1 + len(rows), len(headers), Inches(0.7), Inches(1.8), Inches(12.0), Inches(4.5))
    table = shape.table

    for c, h in enumerate(headers):
        table.cell(0, c).text = h
    for r, row in enumerate(rows, start=1):
        for c, val in enumerate(row[:len(headers)]):
            table.cell(r, c).text = val


def _append_chart_slide(prs: Presentation, title: str, chart_lines):
    chart_type = 'line'
    chart_title = title
    categories = []
    series_name = 'Series'
    values = []

    for ln in chart_lines:
        if ':' not in ln:
            continue
        k, v = ln.split(':', 1)
        key, val = k.strip().lower(), v.strip()
        if key == 'type':
            chart_type = val.lower()
        elif key == 'title':
            chart_title = val
        elif key == 'categories':
            categories = [x.strip() for x in val.split(',') if x.strip()]
        elif key == 'series' and '=' in val:
            sname, sval = val.split('=', 1)
            series_name = sname.strip() or 'Series'
            values = [float(x.strip()) for x in sval.split(',') if x.strip()]

    if not categories or not values or len(categories) != len(values):
        return

    slide = prs.slides.add_slide(prs.slide_layouts[1] if len(prs.slide_layouts) > 1 else prs.slide_layouts[0])
    if slide.shapes.title:
        slide.shapes.title.text = chart_title

    data = CategoryChartData()
    data.categories = categories
    data.add_series(series_name, values)

    chart_map = {
        'line': XL_CHART_TYPE.LINE,
        'bar': XL_CHART_TYPE.BAR_CLUSTERED,
        'column': XL_CHART_TYPE.COLUMN_CLUSTERED,
        'pie': XL_CHART_TYPE.PIE,
    }
    ctype = chart_map.get(chart_type, XL_CHART_TYPE.LINE)

    slide.shapes.add_chart(ctype, Inches(0.8), Inches(1.8), Inches(11.8), Inches(4.8), data)


def generate_ppt_from_markdown(md_text: str, output_name: str = 'chatppt_v0_2.pptx'):
    """
    1) parse_input_text
    2) generate_presentation
    """
    powerpoint_data, presentation_title = parse_input_text(md_text, layout_manager)
    output_path = OUTPUT_DIR / output_name

    generate_presentation(powerpoint_data, str(TEMPLATE_PATH), str(output_path))

    # 最小补充 table/chart
    extra_blocks = extract_table_and_chart_blocks(md_text)
    if extra_blocks:
        prs = Presentation(str(output_path))
        for block in extra_blocks:
            if block['type'] == 'table':
                _append_table_slide(prs, block['title'], block['lines'])
            elif block['type'] == 'chart':
                _append_chart_slide(prs, block['title'], block['lines'])
        prs.save(str(output_path))

    return output_path


print('PPT generator ready (main functions + minimal table/chart patch).')

PPT generator ready (main functions + minimal table/chart patch).


In [10]:
sample_user_input = """
请生成季度复盘PPT：
- 首页概览（2-3条要点）
- 一页图片（images/performance_chart.png）
- 一页地区销售表
- 一页季度趋势图
""".strip()

sample_markdown = """
# 季度业务复盘

## 概览
- 本季度收入环比增长 12%
- 新增客户数提升 18%

## 数据看板
![数据看板](images/performance_chart.png)

## 地区销售表
| 地区 | 销售额 | 增长率 |
|---|---:|---:|
| 华北 | 120 | 10% |
| 华东 | 180 | 18% |
| 华南 | 160 | 15% |

## 季度趋势图
```chart
type: line
title: 季度销售趋势
categories: Q1,Q2,Q3,Q4
series: 销售额=120,150,180,210
```
""".strip()

converted_markdown = format_to_chatppt_markdown(sample_user_input)
print('Converted markdown preview:')
print(converted_markdown[:800])

ppt_path = generate_ppt_from_markdown(sample_markdown, 'chatppt_v0_2_sample.pptx')
print('Generated PPT:', ppt_path)

Converted markdown preview:
# 自动生成演示

## 概览
- 请生成季度复盘PPT：
- 首页概览（2-3条要点）
- 一页图片（images/performance_chart.png）
- 一页地区销售表

## 图片页
![示例图片](images/performance_chart.png)
所有默认幻灯片已被移除。
Presentation saved to 'outputs/chatppt_v0_2_sample.pptx'
Generated PPT: outputs/chatppt_v0_2_sample.pptx


/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/slide1.xml'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/_rels/slide1.xml.rels'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/slide2.xml'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/_rels/slide2.xml.rels'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


In [11]:
def chatbot_convert(user_msg, history):
    history = history or []
    md = format_to_chatppt_markdown(user_msg)
    history.append((user_msg, '已转换为 ChatPPT Markdown，请检查后点击生成。'))
    return history, md, ''


def chatbot_generate(md_text):
    if not md_text or not md_text.strip():
        return '请先输入或转换 Markdown。', None
    out_path = generate_ppt_from_markdown(md_text, 'chatppt_v0_2_gui.pptx')
    return '生成成功。', str(out_path)


with gr.Blocks(title='ChatPPT v0.2 Minimal') as demo:
    gr.Markdown('# ChatPPT v0.2 (Minimal)')
    gr.Markdown('复用 chatppt-main 现有函数，支持 Markdown 转换与 PPT 生成。')

    chatbot = gr.Chatbot(height=320, label='ChatBot')
    user_input = gr.Textbox(lines=6, label='用户输入（自然语言）')

    with gr.Row():
        btn_convert = gr.Button('转换 Markdown', variant='primary')
        btn_generate = gr.Button('生成 PPT')
        btn_clear = gr.Button('清空')

    md_editor = gr.Textbox(lines=18, label='ChatPPT Markdown（可编辑）')
    status = gr.Textbox(label='状态')
    file_out = gr.File(label='下载 PPT')

    btn_convert.click(chatbot_convert, [user_input, chatbot], [chatbot, md_editor, user_input])
    btn_generate.click(chatbot_generate, [md_editor], [status, file_out])
    btn_clear.click(lambda: ([], '', '', None), None, [chatbot, user_input, status, file_out])

demo
# 运行服务：demo.launch(server_name='0.0.0.0', server_port=7860, share=False)

Gradio Blocks instance: 3 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x117f781a0>
 |-<gradio.components.chatbot.Chatbot object at 0x117f78050>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x117f78050>
 |-<gradio.components.textbox.Textbox object at 0x117f70cd0>
 |-<gradio.components.textbox.Textbox object at 0x117f781a0>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x117f70cd0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x117f70e10>
 |-<gradio.components.file.File object at 0x117f78590>
fn_index=2
 inputs:
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x117f78050>
 |-<gradio.components.textbox.Textbox object at 0x117f781a0>
 |-<gradio.components.textbox.Textbox object at 0x117f70e10>
 |-<gradio.components.file.File object at 0x117f78590>

In [12]:
# 最小端到端检查
out = generate_ppt_from_markdown(sample_markdown, 'chatppt_v0_2_test.pptx')
assert Path(out).exists(), 'PPT file was not created'

prs = Presentation(str(out))
assert len(prs.slides) >= 4, 'Expected at least 4 slides'
print('E2E passed:', out, 'slides=', len(prs.slides))

所有默认幻灯片已被移除。
Presentation saved to 'outputs/chatppt_v0_2_test.pptx'


/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/slide1.xml'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/_rels/slide1.xml.rels'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/slide2.xml'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/Users/elon/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py:1642: UserWarning: Duplicate name: 'ppt/slides/_rels/slide2.xml.rels'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


AssertionError: Expected at least 4 slides